In [1]:
# ============================================
# NOTEBOOK 2 — Data Cleaning & Validation
# Sales Intelligence Project
# Analyst: Danish Siddiqui
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

print("=" * 50)
print("   DATA CLEANING & VALIDATION PIPELINE")
print("=" * 50)

# ── STEP 1: Load Raw Data ──────────────────────
sales    = pd.read_csv('data/sales_data.csv')
products = pd.read_csv('data/products.csv')
regions  = pd.read_csv('data/regions.csv')

print(f"\n✅ Raw data loaded: {len(sales)} rows, {sales.shape[1]} columns")

   DATA CLEANING & VALIDATION PIPELINE

✅ Raw data loaded: 502 rows, 8 columns


In [2]:
# ── STEP 2: Check Data Quality Issues ─────────
print("\n📋 INITIAL DATA QUALITY REPORT")
print("-" * 40)
print(f"Total rows        : {len(sales)}")
print(f"Duplicate rows    : {sales.duplicated().sum()}")
print(f"Missing values    :\n{sales.isnull().sum()}")
print(f"\nData types:\n{sales.dtypes}")


📋 INITIAL DATA QUALITY REPORT
----------------------------------------
Total rows        : 502
Duplicate rows    : 0
Missing values    :
order_id         0
date             0
product_id       0
region_id        0
quantity         0
unit_price       0
discount_pct     0
revenue         10
dtype: int64

Data types:
order_id            str
date                str
product_id          str
region_id           str
quantity          int64
unit_price        int64
discount_pct    float64
revenue         float64
dtype: object


In [3]:
# ── STEP 3: Remove Duplicates ─────────────────
before = len(sales)
sales = sales.drop_duplicates()
after  = len(sales)
print(f"✅ Duplicates removed: {before - after} rows dropped")
print(f"   Rows remaining: {after}")

✅ Duplicates removed: 0 rows dropped
   Rows remaining: 502


In [4]:
# ── STEP 4: Fix Missing Values ────────────────
missing_before = sales['revenue'].isnull().sum()

# Fill missing revenue = quantity × unit_price × (1 - discount)
mask = sales['revenue'].isnull()
sales.loc[mask, 'revenue'] = (
    sales.loc[mask, 'quantity'] *
    sales.loc[mask, 'unit_price'] *
    (1 - sales.loc[mask, 'discount_pct'])
).round(2)

print(f"✅ Missing revenue fixed: {missing_before} nulls → 0 nulls")
print(f"   Missing values now: {sales['revenue'].isnull().sum()}")

✅ Missing revenue fixed: 10 nulls → 0 nulls
   Missing values now: 0


In [5]:
# ── STEP 5: Fix Data Types ────────────────────
sales['date']     = pd.to_datetime(sales['date'])
sales['month']    = sales['date'].dt.month
sales['month_name']= sales['date'].dt.strftime('%b')
sales['year']     = sales['date'].dt.year
sales['quarter']  = sales['date'].dt.quarter

print("✅ Date column converted and expanded:")
print(sales[['date','month','month_name','year','quarter']].head(3))

✅ Date column converted and expanded:
        date  month month_name  year  quarter
0 2023-01-01      1        Jan  2023        1
1 2023-01-02      1        Jan  2023        1
2 2023-01-03      1        Jan  2023        1


In [6]:
# ── STEP 6: Merge with Product & Region Info ──
sales = sales.merge(products[['product_id','product_name','category']], 
                    on='product_id', how='left')
sales = sales.merge(regions[['region_id','region_name','zone']], 
                    on='region_id', how='left')

print("✅ Data merged with products and regions")
print(f"   Final shape: {sales.shape}")
print(f"\nSample clean data:")
print(sales.head(3).to_string())

✅ Data merged with products and regions
   Final shape: (502, 16)

Sample clean data:
   order_id       date product_id region_id  quantity  unit_price  discount_pct  revenue  month month_name  year  quarter      product_name     category region_name   zone
0  ORD_0001 2023-01-01       P004       R06         2       35000           0.0  70000.0      1        Jan  2023        1          Tablet X  Electronics        Pune   West
1  ORD_0002 2023-01-02       P005       R04        15        2200           0.0  33000.0      1        Jan  2023        1         USB-C Hub  Accessories     Chennai  South
2  ORD_0003 2023-01-03       P003       R08         8        4500           0.0  36000.0      1        Jan  2023        1  Wireless Earbuds        Audio   Ahmedabad   West


In [8]:
# ── STEP 7: Save Clean Data ───────────────────
sales.to_csv('data/sales_clean.csv', index=False)
print("✅ Clean data saved → data/sales_clean.csv")

print("\n" + "=" * 50)
print("   DATA CLEANING COMPLETE ✅")
print(f"   Final dataset: {len(sales)} rows, {sales.shape[1]} columns")
print(f"   Duplicates removed : YES")
print(f"   Nulls fixed        : YES")
print(f"   Data types fixed   : YES")
print("=" * 50)

✅ Clean data saved → data/sales_clean.csv

   DATA CLEANING COMPLETE ✅
   Final dataset: 502 rows, 16 columns
   Duplicates removed : YES
   Nulls fixed        : YES
   Data types fixed   : YES
